In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

import torch
from torchvision.models import resnet50
import torchvision.transforms as T
torch.set_grad_enabled(False)

# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]


transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# For output bounding box post-processing
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    # Create tensor on the same device as out_bbox to avoid device mismatch
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32, device=out_bbox.device)
    return b

def detect(im, model, transform, device, active_heads=None, active_layers=None):
    # Mean-std normalize the input image (batch-size: 1)
    img = transform(im).unsqueeze(0)

    img = img.to(device)
    
    # Propagate through the model
    if active_heads is not None:
        outputs = model(img, effective_heads=active_heads)
    elif active_layers is not None:
        outputs = model(img, active_layers=active_layers)
    else:
        outputs = model(img)
        
    # Keep only predictions with 0.7+ confidence
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > 0.7

    # Convert boxes from [0; 1] to image scales
    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)
    return probas[keep], bboxes_scaled

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=1)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '1head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '4head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'baseline' / '4head' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from sliced_models import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'sliced' / '300epoch' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import sys
from pathlib import Path
import torch 
# Add project paths
project_root = Path(r'C:\workspace\ml\workspace\master\original')
sys.path.insert(0, str(project_root))

from layer_scaling import build_model
from evaluation.args import HeadArgs

# Initialize custom DETR model with 4 heads
args = HeadArgs(number_of_heads=4)
model, criterion, postprocessors = build_model(args)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_path = project_root / 'results' / 'layer_scaling' / '300epoch' / 'checkpoint.pth'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

model.eval()
print(f"Custom DETR model with {args.nheads} heads loaded on {device}")

In [ ]:
import random
import json
from pathlib import Path
from collections import Counter

# ── Setup COCO dataset ────────────────────────────────────────────────────────
coco_path = 'C:\\workspace\\ml\\data\\coco'
image_set = 'val'
root = Path(coco_path)
assert root.exists(), f'provided COCO path {root} does not exist'

mode = 'instances'
PATHS = {
    "train": (root / "train2017", root / "annotations" / f'{mode}_train2017.json'),
    "val": (root / "val2017", root / "annotations" / f'{mode}_val2017.json'),
}

img_folder, ann_file = PATHS[image_set]

with open(ann_file) as f:
    coco_data = json.load(f)

# ── Select image with selected classes ────────────────────────────────────────
# Find images that contain annotations with selected class indices
category_id_to_idx = {cat['id']: i for i, cat in enumerate(coco_data['categories'])}

valid_image_ids = set()
for ann in coco_data['annotations']:
    cat_idx = category_id_to_idx.get(ann['category_id'])
    if ann['category_id'] in [36]:  # person, car, truck, snowboard
        valid_image_ids.add(ann['image_id'])

if valid_image_ids:
    selected_image_id = random.choice(list(valid_image_ids))
    random_image_info = next(img for img in coco_data['images'] if img['id'] == selected_image_id)
    print(f"Found {len(valid_image_ids)} images with selected classes")
else:
    # Fallback to random image if no images found with selected classes
    print(f"No images found with selected classes, using random image")
    random_image_info = random.choice(coco_data['images'])

file_name = "000000402118.jpg"  # Replace with the desired file name
image_path = img_folder / file_name
im = Image.open(image_path).convert("RGB")

print(f"Selected image: {file_name}")
print(f"Image size: {im.size}")

# ── Run detection ─────────────────────────────────────────────────────────────
scores, boxes = detect(im, model, transform, device)
print(f"Detected {len(scores)} objects with confidence > 0.7")


# ── Plot results ──────────────────────────────────────────────────────────────
def plot_results(pil_img, prob, boxes):
    plt.figure(figsize=(16, 10))
    plt.imshow(pil_img)
    ax = plt.gca()
    bright_red = [1.0, 0.0, 0.0]
    
    for p, (xmin, ymin, xmax, ymax) in zip(prob, boxes.tolist()):
        c = bright_red
        
        # Bounding box
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=1))
        
        cl = p.argmax()
        text = f'{CLASSES[cl]}: {p[cl]:0.2f}'
        ax.text(xmin, ymin, text, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))

        cx = (xmin + xmax) / 2
        cy = (ymin + ymax) / 2
        ax.plot(cx, cy, marker='o', color=c, markersize=6)

        w = xmax - xmin
        h = ymax - ymin
        ax.text(cx, cy + 12, f'x={cx:.1f}, y={cy:.1f}, w={w:.1f}, h={h:.1f}', fontsize=10,
                ha='center', color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=2))

    ax.set_title(f'Detected {len(prob)} objects (filtered by selected classes)', fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_results(im, scores, boxes)

In [ ]:
000000210915.jpg
000000425702.jpg
000000183127.jpg
000000393469.jpg
000000402118.jpg

In [ ]:
coco_data['categories']